# M02-03 — Calidad y limpieza

Referencia de validación. El alumno trabaja en `notebooks/alumno/M02-03-calidad-limpieza.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark("novashop-m02")
from pyspark.sql.functions import col, coalesce, to_timestamp, when, trim
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DecimalType, TimestampType,
)
orders = (
    spark.read.option("header", True).csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
    .withColumn(
        "order_ts",
        coalesce(
            to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
            to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
        ),
    )
    .drop("order_ts_raw")
)
items = (
    spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount", col("discount").cast(DecimalType(5, 2)))
)
customers = spark.read.option("header", True).csv(str(RAW / "customers.csv"))
products = (
    spark.read.option("multiLine", True).json(str(RAW / "products.json"))
    .withColumnRenamed("productId", "product_id")
    .withColumnRenamed("listPrice", "list_price")
    .withColumn("list_price", col("list_price").cast(DecimalType(10, 2)))
)
events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("session_id", StringType(), True),
    StructField("page", StringType(), True),
    StructField("product_id", StringType(), True),
])
events = spark.read.schema(events_schema).json(str(RAW / "events.jsonl"))


## 1–2 — Reglas


In [ ]:
customers_clean = customers.withColumn(
    "country", when(trim(col("country")) == "", "UNK").otherwise(col("country")),
)
products_clean = products.where(col("list_price").isNotNull())
orders_clean = orders.where(trim(col("customer_id")) != "")
items_clean = items.where((trim(col("product_id")) != "") & (col("qty") > 0))
events_clean = events.where(col("customer_id").isNotNull())
print("customers", customers_clean.count(), "unk", customers_clean.where(col("country") == "UNK").count())
print("products", products_clean.count())
print("orders", orders_clean.count(), "items", items_clean.count(), "events", events_clean.count())
assert customers_clean.count() == 250
assert customers_clean.where(col("country") == "UNK").count() == 5
assert products_clean.count() == 57
assert orders_clean.count() == 788
assert items_clean.count() == 2010
assert events_clean.count() == 2420


## 3 — Staging


In [ ]:
STAGING.mkdir(parents=True, exist_ok=True)
pairs = {
    "customers_clean": customers_clean,
    "products_clean": products_clean,
    "orders_clean": orders_clean,
    "order_items_clean": items_clean,
    "events_clean": events_clean,
}
for name, frame in pairs.items():
    dest = STAGING / name
    frame.write.mode("overwrite").parquet(str(dest))
    print(name, spark.read.parquet(str(dest)).count())
assert spark.read.parquet(str(STAGING / "orders_clean")).count() == 788
print("M02-03 OK")
